# Verify IP-Adapter LoRA effect

Compare reconstructions on N held-out HVM test stimuli with and without the LoRA-adapted IP-Adapter, conditioning on **GT SigLIP** (cleanest signal for isolating LoRA's contribution). Outputs a side-by-side grid + SSIM / SigLIP-cosine metrics.

In [ ]:
import sys
from pathlib import Path
import torch
import torchvision.transforms as T
from PIL import Image

sys.path.append('.')
from config_const import (
    SEED, HVM_STIM_DIR, HVM_N_VAR, HVM_CATEGORIES,
    HVM_SIGLIP_EMBEDDINGS_PATH, IP_LORA_DEFAULT_PATH,
)
from data_utils.hvm_loader import _category_stratified_split
from generation.flux_instantx import load_pipeline, generate_img2img
from generation.aperture import load_hvm_packed_aperture_mask
from encoders.siglip_embed import load_siglip, embed_images_stripped
from eval.metrics import ssim, cosine_similarity
from get_device import get_device

DEVICE      = get_device()
N_TEST      = 5
IMAGE_SIZE  = 512
STRENGTH    = 0.6
STEPS       = 20
LORA_PATH   = IP_LORA_DEFAULT_PATH
print(f'device={DEVICE}, lora={LORA_PATH}, exists={Path(LORA_PATH).exists()}')

In [ ]:
# pick N held-out test stimuli (category-stratified)
_, _, test_idx = _category_stratified_split(SEED)
test_idx = test_idx[:N_TEST]

def load_stim(i):
    cat = HVM_CATEGORIES[i // HVM_N_VAR]
    var = i % HVM_N_VAR
    return Image.open(HVM_STIM_DIR / cat / f'{var:02d}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)

originals = [load_stim(int(i)) for i in test_idx]
siglip_gt = torch.load(HVM_SIGLIP_EMBEDDINGS_PATH, weights_only=True)[test_idx]
print(f'{len(originals)} stimuli, siglip_gt {tuple(siglip_gt.shape)}')

In [ ]:
# generate with each pipeline
aperture = load_hvm_packed_aperture_mask(image_size=IMAGE_SIZE, device='cpu', dtype=torch.bfloat16)

def gen(pipe, image_proj, orig, sig, seed):
    return generate_img2img(
        pipe, image_proj, orig, sig,
        ip_adapter_scale=1.0, prompt='', strength=STRENGTH,
        height=IMAGE_SIZE, width=IMAGE_SIZE,
        num_inference_steps=STEPS, guidance_scale=3.5,
        aperture_mask=aperture, seed=seed, show_progress=False,
    )

print('generating frozen baseline...')
pipe, image_proj = load_pipeline(device=DEVICE, default_scale=1.0)
frozen_recons = [gen(pipe, image_proj, orig, sig, i) for i, (orig, sig) in enumerate(zip(originals, siglip_gt))]
del pipe, image_proj; torch.cuda.empty_cache()

print('generating with LoRA...')
pipe, image_proj = load_pipeline(device=DEVICE, default_scale=1.0, ip_lora_path=LORA_PATH)
lora_recons = [gen(pipe, image_proj, orig, sig, i) for i, (orig, sig) in enumerate(zip(originals, siglip_gt))]
del pipe, image_proj; torch.cuda.empty_cache()
print('done')

In [ ]:
# side-by-side grid: original | frozen | lora
from PIL import ImageDraw, ImageFont
labels = ['Original', 'Frozen IP-Adapter', 'LoRA IP-Adapter']
try: font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 18)
except: font = ImageFont.load_default()
W, H = IMAGE_SIZE, IMAGE_SIZE
LH = 32
grid = Image.new('RGB', (3 * W, N_TEST * H + LH), (30, 30, 30))
d = ImageDraw.Draw(grid)
for j, lbl in enumerate(labels):
    bbox = d.textbbox((0, 0), lbl, font=font)
    d.text((j * W + (W - (bbox[2] - bbox[0])) // 2, 6), lbl, font=font, fill=(255, 255, 255))
for r in range(N_TEST):
    for c, im in enumerate([originals[r], frozen_recons[r], lora_recons[r]]):
        grid.paste(im, (c * W, LH + r * H))
out_path = Path('../outputs/verify_ip_lora.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
grid.save(out_path)
print(f'saved {out_path}')
grid

In [ ]:
# metrics: SSIM (vs. original) + SigLIP-cosine (vs. GT embedding)
to_tensor = T.ToTensor()
siglip_proc, siglip_model = load_siglip(device=DEVICE)

def metrics(recons):
    ssims = [ssim(to_tensor(r), to_tensor(o)) for r, o in zip(recons, originals)]
    rec_sigs = embed_images_stripped(siglip_proc, siglip_model, recons)
    coss = cosine_similarity(rec_sigs, siglip_gt).tolist()
    return ssims, coss

fz_s, fz_c = metrics(frozen_recons)
lo_s, lo_c = metrics(lora_recons)

print(f'{"":<20s}{"SSIM↑":>10s}{"SigLIP-cos↑":>14s}')
print(f'{"frozen":<20s}{sum(fz_s)/N_TEST:>10.4f}{sum(fz_c)/N_TEST:>14.4f}')
print(f'{"lora":<20s}{sum(lo_s)/N_TEST:>10.4f}{sum(lo_c)/N_TEST:>14.4f}')
print(f'{"delta":<20s}{(sum(lo_s)-sum(fz_s))/N_TEST:>+10.4f}{(sum(lo_c)-sum(fz_c))/N_TEST:>+14.4f}')